# Notebook 01 — Data Collection

This notebook assembles raw data from two active sources:
1. **IRS 990-PF Index CSVs** (`apps.irs.gov` — no auth, fast)
2. **ProPublica Nonprofit Explorer API** (no auth, used for org profiles and enrichment)

And one optional heavy source:
3. **IRS 990-PF XML bulk ZIPs** (`apps.irs.gov` — no auth, but ~400 MB per year)

> **Note on IRS data:** The IRS deprecated its AWS S3 e-file bucket in December 2021.
> Individual XML files at `s3.amazonaws.com/irs-form-990/` are no longer accessible.
> Data is now distributed as bulk ZIP archives. The index CSVs are still fast and small.

All raw outputs are saved to `data/raw/` and are gitignored.

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import logging
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

from src.api_client import (
    fetch_990_index,
    download_990pf_from_zip,
    load_cached_990pf_xml,
    parse_990pf_grants,
    parse_990pf_funder_summary,
    propublica_organization,
    propublica_search,
)

logging.basicConfig(level=logging.INFO)
RAW = Path('../data/raw')
RAW.mkdir(exist_ok=True)
print('Setup complete.')

## 1  IRS 990-PF: Download Index Files (Fast)

The index CSVs are small (~5-20 MB each) and list every electronically filed 990-PF
for a given year. We use these to identify which foundations filed, their EINs, and
their OBJECT_IDs (needed to locate their XML files in the bulk ZIPs).

Available years with working index CSVs: **2017–2023**.

In [ ]:
YEARS = range(2017, 2024)

all_index_rows = []

for year in YEARS:
    try:
        rows = fetch_990_index(year)
        for r in rows:
            r['INDEX_YEAR'] = year
        all_index_rows.extend(rows)
        print(f"{year}: {len(rows):,} 990-PF filings")
    except Exception as e:
        print(f"{year}: ERROR — {e}")
    time.sleep(0.5)

index_df = pd.DataFrame(all_index_rows)
index_df.to_csv(RAW / '990pf_index.csv', index=False)
print(f"\nTotal index rows saved: {len(index_df):,}")
index_df.head()

In [ ]:
# Quick look at what we have
print(f"Unique foundations (by EIN): {index_df['EIN'].nunique():,}")
print(f"Years covered: {sorted(index_df['INDEX_YEAR'].unique())}")
print(f"\nFilings per year:")
print(index_df.groupby('INDEX_YEAR').size().to_string())

## 2  IRS 990-PF: Download XML Data (Optional — Large Files)

The actual grant-level data is in 990-PF XML files, which are only available as
bulk ZIP archives (~400 MB each). This cell downloads **one chunk** for 2020 and
extracts a sample of 990-PF XMLs to parse for grant records.

> **Skip this cell** if you want to proceed with ProPublica-only data.
> The full analysis pipeline works without it — grant detail will be limited
> to what's in the index CSV (totals, not individual recipients).

In [ ]:
DOWNLOAD_XML = False  # Set to True to download ~400 MB ZIP

grants_records = []
funder_records = []

if DOWNLOAD_XML:
    # Get ObjectIds from the 2020 index for a sample of foundations
    sample_ids = index_df[index_df['INDEX_YEAR'] == 2020]['OBJECT_ID'].dropna().tolist()[:500]
    print(f"Targeting {len(sample_ids)} ObjectIds from 2020")

    # First check local cache (avoids re-downloading)
    cached = {}
    for oid in sample_ids:
        xml = load_cached_990pf_xml(oid)
        if xml:
            cached[oid] = xml
    print(f"Found {len(cached)} already cached, need to download {len(sample_ids) - len(cached)}")

    remaining = [oid for oid in sample_ids if oid not in cached]
    downloaded = dict(cached)

    if remaining:
        # Download chunk 1 of 2020 (~400 MB) and extract matching files
        chunk_results = download_990pf_from_zip(2020, remaining, chunk='1', save=True)
        downloaded.update(chunk_results)

    # Parse all available XMLs
    for oid, xml_text in tqdm(downloaded.items(), desc='Parsing XMLs'):
        try:
            grants = parse_990pf_grants(xml_text)
            for g in grants:
                g['object_id'] = oid
            grants_records.extend(grants)
            funder_records.append(parse_990pf_funder_summary(xml_text))
        except Exception as e:
            print(f"Parse error {oid}: {e}")

    print(f"\nParsed: {len(funder_records)} filings, {len(grants_records)} grant records")
else:
    print("XML download skipped. Set DOWNLOAD_XML = True to enable.")
    print("Continuing with index-level data and ProPublica enrichment.")

In [ ]:
import pandas as pd

# Save whatever we have (empty DataFrames if XML was skipped)
grants_raw = pd.DataFrame(grants_records)
funders_raw = pd.DataFrame(funder_records)

grants_raw.to_csv(RAW / 'grants_raw.csv', index=False)
funders_raw.to_csv(RAW / 'funders_raw.csv', index=False)

print(f"grants_raw: {grants_raw.shape}")
print(f"funders_raw: {funders_raw.shape}")
if not grants_raw.empty:
    print(grants_raw.head())

## 3  ProPublica: Build Foundation List and Enrich with Org Profiles

We use two ProPublica strategies:
- **From index**: Look up orgs by EIN (for foundations we found in the IRS index)
- **From search**: Search for known large private foundations to seed the dataset

ProPublica returns NTEE codes, total revenue, state, city, and filing history.

In [ ]:
# Seed list: EINs of major private foundations known for racial equity funding
# (Ford, MacArthur, Open Society, Kellogg, Mellon, Gates, JPB, Rockefeller, Walton, Bloomberg)
SEED_EINS = [
    '131684331',  # Ford Foundation
    '362424426',  # MacArthur Foundation
    '133196841',  # Open Society Foundations
    '381359217',  # W.K. Kellogg Foundation
    '131879673',  # Andrew W. Mellon Foundation
    '562618866',  # Bill & Melinda Gates Foundation
    '270952093',  # JPB Foundation
    '131659629',  # Rockefeller Foundation
    '710578642',  # Walton Family Foundation
    '510397498',  # Bloomberg Philanthropies
    '946064510',  # Hewlett Foundation
    '941655673',  # David & Lucile Packard Foundation
    '316001440',  # Robert Wood Johnson Foundation
    '526051332',  # Annie E. Casey Foundation
    '042103547',  # Hyams Foundation
]

# Also sample EINs from the IRS index (larger foundations by filing)
index_eins = index_df['EIN'].dropna().unique().tolist()[:200]
all_eins = list(set(SEED_EINS + index_eins))
print(f"Looking up {len(all_eins)} foundation EINs via ProPublica")

In [ ]:
foundation_records = []

for ein in tqdm(all_eins, desc='ProPublica lookups'):
    try:
        data = propublica_organization(ein)
        org = data.get('organization', {})
        filings = data.get('filings_with_data', [])
        foundation_records.append({
            'ein': ein,
            'name': org.get('name'),
            'ntee_code': org.get('ntee_code'),
            'state': org.get('state'),
            'city': org.get('city'),
            'total_revenue': org.get('revenue_amount'),
            'subsection_code': org.get('subsection_code'),
            'num_filings': len(filings),
            'latest_filing_year': filings[0].get('tax_prd_yr') if filings else None,
        })
        time.sleep(0.3)
    except Exception as e:
        foundation_records.append({'ein': ein, 'error': str(e)})

foundations_pp = pd.DataFrame(foundation_records)
foundations_pp.to_csv(RAW / 'foundations_propublica.csv', index=False)
print(f"Enriched {foundations_pp['name'].notna().sum()} foundations")
foundations_pp[foundations_pp['name'].notna()].head(10)

## 4  Candid APIs (Stub)

These cells activate once `CANDID_API_KEY` is set in a `.env` file at the project root.

In [ ]:
# Uncomment after setting CANDID_API_KEY in .env

# from dotenv import load_dotenv
# load_dotenv('../.env')
# from src.api_client import candid_demographics, candid_grants_search
#
# demo_records = []
# for ein in tqdm(all_eins[:200], desc='Candid demographics'):
#     try:
#         data = candid_demographics(ein)
#         demo_records.append(data)
#         time.sleep(0.5)
#     except Exception as e:
#         print(f'  {ein}: {e}')
# pd.DataFrame(demo_records).to_csv(RAW / 'demographics_raw.csv', index=False)

print('Candid API stub — set CANDID_API_KEY in .env to activate')

## Summary

Data saved to `data/raw/`:
- `990pf_index.csv` — IRS index of all 990-PF filings (fast, ~7 years)
- `foundations_propublica.csv` — ProPublica org profiles with NTEE codes
- `grants_raw.csv` — Individual grant records (populated if DOWNLOAD_XML = True)
- `funders_raw.csv` — Funder summaries from XML parsing

Proceed to **Notebook 02** for cleaning and SQLite loading.